# W14-D7 Virtual CTO Review（执行版）：Semantic Model v0.1 机器质检 + 评分稳健性

与 md（评审叙事）差异化：本 notebook 是**评审结论的可执行复现**——
1. Cell 2：12 项一致性检查直接跑在真实文件上（定稿包 YAML × D3 体检 × D5 覆盖率 × ontology 指纹）
2. Cell 3：盲区×稀疏四象限（用 D3/D5 真实数据，修正"盲区=稀疏"的旧断言）
3. Cell 4：五维评分稳健性蒙特卡洛——分数是裁决不是测量

全部只读真实文件，不修改任何源。

In [ ]:
# 字体配置：TOOLS.md 唯一标准（所有 notebook 必须用这个）
from matplotlib import font_manager
font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"

font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()

import matplotlib.pyplot as plt
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

print("使用字体:", font_name)

## Cell 2：12 项一致性机器检查（评审 §3 的证据）

检查对象：定稿包 v0.1 是否与它的两份证据报告（D3/D5）和上游四源自洽。
这正是 D3 提出的"体检可复现原则"的第一次实战：评审不读数字，评审**重算**数字。

In [ ]:
import yaml, hashlib, os

SM = '/root/learning-notebooks/semantic-model/mi-cre-semantic-model-v0.1.yaml'
D3 = '/root/learning-notebooks/第14周/w14d3-ontology-health-report.yaml'
D5 = '/root/learning-notebooks/第14周/w14d5-context-coverage-report.yaml'
ONT = '/root/docs/lanlnk/config/ontology/business-ontology.yaml'

sm, d3, d5 = yaml.safe_load(open(SM)), yaml.safe_load(open(D3)), yaml.safe_load(open(D5))
checks = []
ck = lambda n, ok, d='': checks.append((n, bool(ok), d))

ctx = sm['entity']['contexts']
s = sum(v['tables'] for v in ctx.values())
ck('C01 YAML 可解析', True)
ck('C02 表数守恒 Σ=472', s == sm['entity']['table_total'], f'{s}')
d5map = {c['context']: (c['tables'], c['share_pct']) for c in d5['by_context']}
ck('C03 与 D5 逐 Context 表数一致', all(d5map[k][0]==v['tables'] for k,v in ctx.items()))
ck('C04 share_pct 复算一致', all(abs(d5map[k][1]-v['share_pct'])<=0.05 for k,v in ctx.items()))
t = sm['identity']['term_layer']
ck('C05 术语层 883/792/792+91 与 D3 一致', (t['entries'],t['unique_terms'],t['reuse_without_disambiguation'])==(883,792,91))
cp = sm['capability_policy']
ck('C06 能力/场景数字与 D3 一致', cp['modules']==d3['counts']['modules'] and cp['scenario_entries']==d3['counts']['scenario_entries'])
gaps = sm['known_gaps']; p0=[g for g in gaps if g.get('severity')=='P0']
ck('C08 缺口 10 条且 P0 唯一(G-01)', len(gaps)==10 and len(p0)==1 and p0[0]['id']=='G-01')
h = hashlib.sha256(open(ONT,'rb').read()).hexdigest()[:16]
ck('C09 ontology 指纹可复现(未漂移)', h==sm['sources']['ontology']['sha256_16'], h)
ck('C10 熵增基线与 D5 月度增长一致', sm['entropy_baseline']['table_growth_monthly']=={m:v['new_tables'] for m,v in d5['monthly_growth'].items()})
fc = sm.get('first_consumer', {})
ck('C11 消费方声明+预警存在', 'planned' in fc and 'warning' in fc)
paths = [sm['sources'][k] for k in ('health_report','coverage_report','rule_audit','effect_registry','canonical_tables')]
ck('C12 证据指针 5/5 存在', all(os.path.exists(p) for p in paths))

# C07 有意留空：名称漂移检查在 Cell 3 深挖（D3 短名 vs D5 全名正是漂移本身）
for n, ok, d in checks:
    print(('PASS' if ok else 'FAIL'), n, d)
print(f'\n== {sum(ok for _,ok,_ in checks)}/{len(checks)} 通过；C07 的教训见 Cell 3 ==')

## Cell 3：盲区×稀疏四象限——修正一个过度概括

D5 journal 曾断言"稀疏 Context 与孤儿 Context 精确重叠"。用真实数据画四象限：
- **x = 表数**（D5，log 轴），**y = 模块入口数**（D3 module_to_contexts 反转求 fan-in）
- 盲区 = y=0（ontology 无模块入口）；稀疏 = x≤3

看图说话：四个孤儿里 12 Engineering(46 表)/16 Parking(30 表)/02 Merchant(22 表) 是**大块头盲区**——
代码长起来了、语义没跟上（定向腐蚀）；只有 15 Customer/Member 既盲又稀疏；
13 WorkOrder(3 表) 相反：**有入口但代码没长**。两种病，两种药。
（注意：D3 用短名 "02 Party Core"、D5 用全名 "02 Merchant"——归一化靠编号前缀，这个漂移本身就是 G-04 现行为证）

In [ ]:
import re
# fan-in：反转 D3 module_to_contexts（D3 短名 → 编号前缀归一化，与 D5 全名对齐）
fan_in = {}
for mod, ctxs in d3['module_to_contexts'].items():
    for c in ctxs:
        num = re.match(r'(\d+)', c)
        if num: fan_in[num.group(1)] = fan_in.get(num.group(1), 0) + 1

# D5 全名 → (表数, fan-in)；Platform/Shared 是跨切面桶不是领域 Context，排除
pts = []
for c in d5['by_context']:
    m = re.match(r'(\d+)', c['context'])
    if not m: continue
    pts.append((c['context'], int(m.group(1)), c['tables'], fan_in.get(m.group(1), 0)))

fig, ax = plt.subplots(figsize=(10, 6.5))
blind_y, sparse_x = 0.5, 3
ax.axhspan(-0.4, blind_y, color='#fdecea', zorder=0)                      # 盲区带
for name, num, tables, fi in pts:
    ax.scatter(tables, fi, s=60+tables*1.6, alpha=.75,
               color=('#d62728' if fi==0 else '#1f77b4'), zorder=3)
    off = (6, 6) if tables < 20 else (-8, 8)
    ax.annotate(f'{name}\n{tables}表', (tables, fi), textcoords='offset points',
                xytext=off, ha=('left' if tables<20 else 'right'), fontsize=8.5)
ax.axvline(sparse_x, ls='--', c='gray', lw=1)
ax.axhline(blind_y, ls='--', c='#d62728', lw=1)
ax.set_xscale('log'); ax.set_xticks([1,3,10,30,100]); ax.set_xticklabels(['1','3','10','30','100'])
ax.set_yticks([0,1,2]); ax.set_xlim(1, 130); ax.set_ylim(-0.4, 2.5)
ax.set_xlabel('表数（log，D5 覆盖率报告）'); ax.set_ylabel('ontology 模块入口数（fan-in）')
ax.set_title('W14-D7 盲区×稀疏四象限：盲区 ≠ 稀疏（红=无模块入口）')
ax.text(60, 0.08, '大块头盲区：代码长大、语义没跟上\n(12 Engineering 46表 / 16 Parking 30表)', fontsize=9, color='#d62728')
ax.text(1.2, 2.2, '反向稀疏：语义认了、代码没长\n(13 WorkOrder 3表 有入口)', fontsize=9, color='#1f77b4')
ax.text(1.2, -0.25, '既盲又稀疏：15 Customer/Member', fontsize=9, color='#d62728')
plt.tight_layout(); plt.savefig('/root/learning-notebooks/第14周/w14d7_盲区稀疏四象限.png', dpi=130); plt.close()

blind = sorted([(n,t) for n,_,t,f in pts if f==0], key=lambda x:-x[1])
print('盲区 Context（按表数降序）:', blind)
print('稀疏 Context(≤3表):', [(n,t,f) for n,_,t,f in pts if t<=3])
print('结论：盲区≠稀疏 ——', sum(1 for _,t in blind if t>3), '个大块头盲区 vs', sum(1 for _,_,t,f in pts if t<=3 and f>0), '个有入口的稀疏区')

## Cell 4：五维评分稳健性蒙特卡洛——分数是裁决不是测量

评审给分（md §6）：AQ=7.5, CH=7.0, ADR=6.0, TD=8.0, DX=5.5，综合 6.4。
这个综合分**稳健吗**？两问：
- **E1 权重扰动**：权重 ~ Dirichlet(1,1,1,1,1) 抽 3 万组 → 综合分分布多宽？各维度权重与综合分的相关性（谁在主导）？
- **E2 木桶锚稳健性**：每个维度独立扰动 ±0.5，DX 仍是最低维的概率是多少？（木桶规则下它决定"下限分"）

这是决策分析玩具，不是测量——它的用途是提醒：报"6.4"时必须同时报规则（均值/木桶）。

In [ ]:
import numpy as np
rng = np.random.default_rng(42)
dims = ['Architecture','CodeHealth','ADR一致','TechDebt','DX']
scores = np.array([7.5, 7.0, 6.0, 8.0, 5.5])

# E1：权重扰动 → 综合分分布
W = rng.dirichlet(np.ones(5), size=30000)
comp = W @ scores
lo, hi = np.percentile(comp, [16, 84])
corr = [round(float(np.corrcoef(W[:,i], comp)[0,1]), 3) for i in range(5)]

# E2：每维 ±0.5 扰动，DX 仍是 argmin 的概率
P = rng.uniform(-0.5, 0.5, size=(30000, 5))
S2 = scores + P
p_dx_min = float((S2.argmin(axis=1) == 4).mean())
p_any_gap = float((S2.min(axis=1) < 6.0).mean())   # 木桶分跌破 6.0 的概率

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8))
ax = axes[0]
ax.hist(comp, bins=60, color='#1f77b4', alpha=.8)
for v, c, lab in [(lo,'#1f77b4',f'16%={lo:.2f}'), (hi,'#1f77b4',f'84%={hi:.2f}')]:
    ax.axvline(v, ls='--', c=c, lw=1); ax.text(v, ax.get_ylim()[1]*.92, lab, ha='center', fontsize=9)
ax.axvline(scores.min(), c='#d62728', lw=2)
ax.text(scores.min(), ax.get_ylim()[1]*.75, f'木桶规则={scores.min():.1f}\n(恒锚 DX)', color='#d62728', ha='center', fontsize=10)
ax.axvline(6.4, c='k', ls=':', lw=1.5); ax.text(6.42, ax.get_ylim()[1]*.5, '评审报分 6.4', fontsize=9, rotation=90)
ax.set_title(f'E1 权重扰动下综合分分布（68% 区间 [{lo:.2f}, {hi:.2f}]）'); ax.set_xlabel('综合分'); ax.set_ylabel('频数')
ax = axes[1]
ax.bar(dims, corr, color=['#1f77b4']*4+['#d62728'])
ax.set_title('E1 维度权重→综合分 相关系数（谁主导均值分）'); ax.set_ylabel('Pearson r')
ax.tick_params(axis='x', labelrotation=20)
for i, v in enumerate(corr): ax.text(i, v+0.01, f'{v:.2f}', ha='center', fontsize=9)
plt.tight_layout(); plt.savefig('/root/learning-notebooks/第14周/w14d7_评分稳健性.png', dpi=130); plt.close()

print(f'E1 综合分 mean={comp.mean():.2f}, 68%区间=[{lo:.2f},{hi:.2f}], 主导维度={dims[int(np.argmax(corr))]}')
print(f'E2 DX 仍为最低维概率={p_dx_min:.1%}；木桶分跌破6.0概率={p_any_gap:.1%}')
print('结论：均值规则下分数随权重摆动 ±0.35；木桶规则稳定锚在 DX —— 排期用木桶，汇报用均值，报分必报规则。')

## 结论（三行）

1. **一致性**：11/12 过 + 1 个失败项揪出语义资产自己的名称漂移（G-04 现行为证）→ v0.1.1 别名表整改
2. **盲区≠稀疏**：12/16/02 是大块头盲区（代码长大、语义没跟上），13 是反向稀疏——两种病两种药
3. **评分是裁决**：综合 6.4 在均值规则下稳健（±0.35），木桶规则恒锚 DX=5.5——W15 挣的就是这一分

全部检查可在任何时点重跑：源文件变 → C09 指纹断 → 熵增量可见。这就是"体检可复现"的机器版。